# Research: Dual Momentum (Gary Antonacci)

## Contexte

La stratégie **Dual Momentum** est décrite dans le livre *"Dual Momentum Investing"* de Gary Antonacci (2014).
C'est l'une des stratégies momentum les plus académiquement robustes, combinant deux dimensions:

- **Momentum absolu** (Time-Series Momentum): Compare chaque actif à son propre historique
  - Si le rendement 12 mois est positif → rester investi
  - Si négatif → aller en refuge (bonds)
- **Momentum relatif** (Cross-Section Momentum): Parmi les actifs avec momentum positif, choisir le meilleur

## Performance actuelle (baseline)

- Stratégie: nouvelle (pas encore de backtest)
- Sharpe attendu: 0.7-1.0 (d'après la littérature)
- Période: 2015-2026

## Objectif de ce notebook

1. Valider la thèse avec yfinance (SPY/EFA/BND)
2. Tester différentes variantes (lookback, seuil absolu)
3. Analyser les régimes (bull/bear) pour comprendre l'edge
4. Formuler la configuration optimale pour l'implémentation QC

## 1. Setup et Données

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Universe
TICKERS = ['SPY', 'EFA', 'BND']
START = '2014-01-01'  # Extra year for warmup
END = '2026-01-01'

print("Chargement des données yfinance...")
raw = yf.download(TICKERS, start=START, end=END, auto_adjust=True)
prices = raw['Close'].dropna()
print(f"Période: {prices.index[0].date()} -> {prices.index[-1].date()}")
print(f"Observations: {len(prices)} jours")
print()
prices.tail(3)

## 2. Analyse Exploratoire

In [ ]:
returns = prices.pct_change().dropna()

print("=== Statistiques annualisées (2015-2026) ===")
stats_period = returns['2015':]
annual_ret = stats_period.mean() * 252
annual_vol = stats_period.std() * np.sqrt(252)
sharpe = annual_ret / annual_vol

stats = pd.DataFrame({
    'CAGR (%)': (annual_ret * 100).round(2),
    'Volatilité (%)': (annual_vol * 100).round(2),
    'Sharpe': sharpe.round(3),
})
print(stats)

print("\n=== Corrélations (rendements journaliers) ===")
print(returns['2015':].corr().round(3))

# Cumulative returns
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax1 = axes[0]
(1 + returns['2015':]).cumprod().plot(ax=ax1)
ax1.set_title('Rendements cumulés 2015-2026 (Buy & Hold)')
ax1.set_ylabel('Valeur du portefeuille (base 1)')

ax2 = axes[1]
returns['2015':].rolling(63).std().mul(np.sqrt(252)).plot(ax=ax2)
ax2.set_title('Volatilité réalisée 63j (annualisée)')
ax2.set_ylabel('Volatilité')

plt.tight_layout()
plt.show()

## 3. Hypothèse 1: Momentum 12 mois comme signal

Tester si le rendement des 12 derniers mois prédit le rendement du mois suivant.
C'est le fondement académique du momentum (Jegadeesh & Titman 1993, Fama & French 1996).

In [ ]:
# Calcul des rendements 12 mois glissants (mensuel)
monthly_prices = prices.resample('ME').last()
monthly_returns = monthly_prices.pct_change()

# Signal: rendement des 12 derniers mois
momentum_12m = monthly_prices.pct_change(12)  # 12 mois

print("=== Distribution des rendements 12m momentum ===")
print(momentum_12m['2015':].describe().round(3))

# Test: Est-ce que momentum_12m > 0 prédit un rendement positif le mois suivant?
for ticker in ['SPY', 'EFA']:
    sig = momentum_12m[ticker]['2015':]
    fwd = monthly_returns[ticker]['2015':].shift(-1)  # rendement du mois suivant
    df_test = pd.DataFrame({'signal': sig, 'forward': fwd}).dropna()
    
    positive_signal = df_test[df_test['signal'] > 0]['forward']
    negative_signal = df_test[df_test['signal'] <= 0]['forward']
    
    print(f"\n{ticker}:")
    print(f"  Signal positif (n={len(positive_signal)}): rendement moyen = {positive_signal.mean():.2%}")
    print(f"  Signal negatif (n={len(negative_signal)}): rendement moyen = {negative_signal.mean():.2%}")
    print(f"  Predictive ratio: {positive_signal.mean() / abs(negative_signal.mean()) if negative_signal.mean() != 0 else 'N/A':.2f}x")

**Verdict Hypothèse 1**: A confirmer par les résultats — si rendement moyen lors de signal positif > signal négatif, la thèse momentum est validée.

## 4. Backtest complet: Dual Momentum 2015-2026

In [ ]:
def backtest_dual_momentum(monthly_prices, lookback_months=12, verbose=False):
    """
    Dual Momentum Backtest (Antonacci)
    - monthly rebalance
    - 100% concentration in chosen asset
    - transaction cost: 0.05% per trade
    """
    risky = ['SPY', 'EFA']
    safe = 'BND'
    TC = 0.0005  # 5 bps transaction cost
    
    portfolio_value = [1.0]
    holdings = []
    signals = []
    
    monthly_ret = monthly_prices.pct_change()
    
    current_holding = None
    
    for i in range(lookback_months + 1, len(monthly_prices)):
        date = monthly_prices.index[i]
        
        # Compute lookback returns for risky assets
        lookback_start = i - lookback_months
        price_now = monthly_prices.iloc[i - 1]  # end of previous month (signal date)
        price_past = monthly_prices.iloc[lookback_start - 1]
        
        returns_12m = {t: (price_now[t] / price_past[t]) - 1 for t in risky}
        
        # Relative momentum: best risky asset
        best_risky = max(returns_12m, key=returns_12m.get)
        best_return = returns_12m[best_risky]
        
        # Absolute momentum: is best risky > 0?
        if best_return > 0:
            target = best_risky
        else:
            target = safe
        
        signals.append({'date': date, 'SPY_12m': returns_12m['SPY'], 
                        'EFA_12m': returns_12m['EFA'], 'target': target})
        
        # Return for this month
        month_ret = monthly_ret[target].iloc[i]
        
        # Apply transaction cost if switching
        tc = TC if target != current_holding else 0
        current_holding = target
        
        pv = portfolio_value[-1] * (1 + month_ret - tc)
        portfolio_value.append(pv)
        holdings.append(target)
        
        if verbose:
            print(f"{date.strftime('%Y-%m')}: SPY={returns_12m['SPY']:.1%}, EFA={returns_12m['EFA']:.1%} -> {target} ({month_ret:.1%})")
    
    # Build portfolio series
    dates = monthly_prices.index[lookback_months + 1:]
    pv_series = pd.Series(portfolio_value[1:], index=dates)
    
    return pv_series, pd.DataFrame(signals).set_index('date'), holdings


# Run backtest on 2015-2026
monthly_prices_2015 = monthly_prices['2015':]
pv, signals_df, holdings = backtest_dual_momentum(monthly_prices, lookback_months=12)
pv_2015 = pv['2015':]

# Benchmark: SPY Buy & Hold
spy_bh = (1 + monthly_prices['SPY'].pct_change()['2015':]).cumprod()

print(f"Portfolio values: {len(pv_2015)} months")
print(f"Strategy final value: {pv_2015.iloc[-1]:.3f}x")
print(f"SPY B&H final value: {spy_bh.iloc[-1]:.3f}x")

In [ ]:
def compute_metrics(pv_series, rf_rate=0.02):
    """Compute key performance metrics from monthly portfolio value series."""
    monthly_ret = pv_series.pct_change().dropna()
    
    n_years = len(monthly_ret) / 12
    cagr = (pv_series.iloc[-1] / pv_series.iloc[0]) ** (1 / n_years) - 1
    
    ann_vol = monthly_ret.std() * np.sqrt(12)
    sharpe = (cagr - rf_rate) / ann_vol if ann_vol > 0 else 0
    
    # Max drawdown
    rolling_max = pv_series.cummax()
    drawdowns = (pv_series - rolling_max) / rolling_max
    max_dd = drawdowns.min()
    
    # Sortino
    negative_returns = monthly_ret[monthly_ret < 0]
    downside_vol = negative_returns.std() * np.sqrt(12) if len(negative_returns) > 0 else 0
    sortino = (cagr - rf_rate) / downside_vol if downside_vol > 0 else 0
    
    return {
        'CAGR': cagr,
        'Volatilité': ann_vol,
        'Sharpe': sharpe,
        'Sortino': sortino,
        'Max DD': max_dd,
        'Calmar': cagr / abs(max_dd) if max_dd != 0 else 0
    }


# Metrics
dm_metrics = compute_metrics(pv_2015)
spy_metrics = compute_metrics(spy_bh.reindex(pv_2015.index, method='ffill'))

metrics_df = pd.DataFrame({
    'Dual Momentum': dm_metrics,
    'SPY B&H': spy_metrics
}).T

for col in ['CAGR', 'Volatilité', 'Max DD']:
    metrics_df[col] = metrics_df[col].map('{:.2%}'.format)
for col in ['Sharpe', 'Sortino', 'Calmar']:
    metrics_df[col] = metrics_df[col].map('{:.3f}'.format)

print("=== Résultats du Backtest Dual Momentum (2015-2026) ===")
print(metrics_df)

In [ ]:
# Visualisation complète
fig, axes = plt.subplots(3, 1, figsize=(16, 14))

# 1. Cumulative performance
ax1 = axes[0]
pv_2015_norm = pv_2015 / pv_2015.iloc[0]
spy_norm = spy_bh.reindex(pv_2015.index, method='ffill')
spy_norm = spy_norm / spy_norm.iloc[0]

pv_2015_norm.plot(ax=ax1, label='Dual Momentum', linewidth=2, color='steelblue')
spy_norm.plot(ax=ax1, label='SPY B&H', linewidth=1.5, color='orange', linestyle='--')
ax1.set_title('Dual Momentum vs SPY (2015-2026)', fontsize=14)
ax1.set_ylabel('Valeur (base 1)')
ax1.legend()

# 2. Holdings over time
ax2 = axes[1]
signals_2015 = signals_df['2015':]
color_map = {'SPY': 'green', 'EFA': 'blue', 'BND': 'gray'}
for date, row in signals_2015.iterrows():
    ax2.axvspan(date, date + pd.DateOffset(months=1), 
                color=color_map.get(row['target'], 'white'), alpha=0.5)

legend_elements = [Patch(facecolor='green', alpha=0.5, label='SPY (US Equities)'),
                   Patch(facecolor='blue', alpha=0.5, label='EFA (Intl Equities)'),
                   Patch(facecolor='gray', alpha=0.5, label='BND (Bonds - Defensive)')]
ax2.legend(handles=legend_elements, loc='center left')
ax2.set_title('Holdings par mois', fontsize=14)
ax2.set_ylabel('Position')
ax2.set_yticks([])

# 3. Drawdown comparison
ax3 = axes[2]
dd_dm = (pv_2015_norm / pv_2015_norm.cummax() - 1) * 100
dd_spy = (spy_norm / spy_norm.cummax() - 1) * 100
dd_dm.plot(ax=ax3, label='Dual Momentum', color='steelblue', linewidth=2)
dd_spy.plot(ax=ax3, label='SPY B&H', color='orange', linewidth=1.5, linestyle='--')
ax3.fill_between(dd_dm.index, dd_dm, 0, alpha=0.2, color='steelblue')
ax3.set_title('Drawdown Comparison', fontsize=14)
ax3.set_ylabel('Drawdown (%)')
ax3.legend()

plt.tight_layout()
plt.show()

## 5. Hypothèse 2: Robustesse par lookback period

La littérature (Antonacci) utilise 12 mois. Tester 6, 9, 12, 24 mois pour vérifier la robustesse.

In [ ]:
lookbacks = [3, 6, 9, 12, 18, 24]
results = {}

for lb in lookbacks:
    pv, _, _ = backtest_dual_momentum(monthly_prices, lookback_months=lb)
    pv_slice = pv['2015':]
    if len(pv_slice) > 12:
        m = compute_metrics(pv_slice)
        results[lb] = m

results_df = pd.DataFrame(results).T
results_df.index.name = 'Lookback (mois)'

print("=== Sensibilité au lookback (2015-2026) ===")
display_df = results_df.copy()
for col in ['CAGR', 'Volatilité', 'Max DD']:
    display_df[col] = display_df[col].map('{:.2%}'.format)
for col in ['Sharpe', 'Sortino', 'Calmar']:
    display_df[col] = display_df[col].map('{:.3f}'.format)
print(display_df)

# Plot Sharpe by lookback
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
results_df['Sharpe'].plot(kind='bar', ax=axes[0], color='steelblue', title='Sharpe Ratio par Lookback')
axes[0].set_xlabel('Lookback (mois)')
axes[0].axhline(y=0, color='r', linestyle='--')

results_df['Max DD'].abs().plot(kind='bar', ax=axes[1], color='salmon', title='Max Drawdown par Lookback (abs)')
axes[1].set_xlabel('Lookback (mois)')

plt.tight_layout()
plt.show()

**Verdict Hypothèse 2**: Si le Sharpe est stable autour de 12 mois (±3 mois), cela confirme la robustesse du paramètre.

## 6. Hypothèse 3: Analyse des régimes (bull vs bear)

La Dual Momentum devrait particulièrement exceller en période de stress, grâce au filtre absolu.

In [ ]:
# Analyse par regime
pv_main, signals_main, _ = backtest_dual_momentum(monthly_prices, lookback_months=12)
signals_2015 = signals_main['2015':]
pv_2015 = pv_main['2015':]

# Identifier les mois défensifs (BND)
defensive_months = signals_2015[signals_2015['target'] == 'BND'].index
spy_months = signals_2015[signals_2015['target'] == 'SPY'].index
efa_months = signals_2015[signals_2015['target'] == 'EFA'].index

total = len(signals_2015)
print(f"=== Distribution des positions (2015-2026) ===")
print(f"SPY (US Equities):    {len(spy_months):3d} mois ({len(spy_months)/total:.1%})")
print(f"EFA (Intl Equities):  {len(efa_months):3d} mois ({len(efa_months)/total:.1%})")
print(f"BND (Bonds):          {len(defensive_months):3d} mois ({len(defensive_months)/total:.1%})")
print()

# Analyse des périodes défensives
print("=== Périodes défensives (BND) ===")
if len(defensive_months) > 0:
    # Regrouper en périodes continues
    defensive_dates = sorted(defensive_months)
    periods = []
    start_period = defensive_dates[0]
    prev_date = defensive_dates[0]
    for d in defensive_dates[1:]:
        if (d - prev_date).days > 40:
            periods.append((start_period, prev_date))
            start_period = d
        prev_date = d
    periods.append((start_period, prev_date))
    
    for start, end in periods:
        # SPY return during that defensive period
        spy_period = monthly_prices['SPY'].loc[start:end].pct_change().sum()
        bnd_period = monthly_prices['BND'].loc[start:end].pct_change().sum()
        print(f"  {start.strftime('%Y-%m')} -> {end.strftime('%Y-%m')}: SPY={spy_period:.1%}, BND={bnd_period:.1%}")

In [ ]:
# Analyse mensuelle: comparaison DM vs SPY pendant les stress periods
monthly_pv = pv_2015.pct_change().dropna()
monthly_spy = monthly_prices['SPY']['2015':].pct_change().dropna().reindex(monthly_pv.index)

# Pendant les mois où SPY < -5%
crash_months = monthly_spy[monthly_spy < -0.05]
if len(crash_months) > 0:
    dm_during_crash = monthly_pv.reindex(crash_months.index).dropna()
    print("=== Mois de crash SPY (< -5%) ===")
    print(f"Nombre de mois: {len(crash_months)}")
    print(f"SPY moyen: {crash_months.mean():.2%}")
    print(f"DM moyen:  {dm_during_crash.mean():.2%}")
    print(f"Protection: {dm_during_crash.mean() - crash_months.mean():.2%}")

# Rolling annual Sharpe
rolling_sharpe_dm = monthly_pv.rolling(12).apply(
    lambda x: (x.mean() * 12 - 0.02) / (x.std() * np.sqrt(12)) if x.std() > 0 else 0
)
rolling_sharpe_spy = monthly_spy.rolling(12).apply(
    lambda x: (x.mean() * 12 - 0.02) / (x.std() * np.sqrt(12)) if x.std() > 0 else 0
)

fig, ax = plt.subplots(figsize=(14, 5))
rolling_sharpe_dm.plot(ax=ax, label='Dual Momentum', linewidth=2, color='steelblue')
rolling_sharpe_spy.plot(ax=ax, label='SPY B&H', linewidth=1.5, color='orange', linestyle='--')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.axhline(y=0.5, color='green', linestyle=':', linewidth=1, label='Sharpe=0.5 target')
ax.set_title('Sharpe Ratio glissant 12 mois', fontsize=14)
ax.set_ylabel('Sharpe Ratio')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Hypothèse 4: Seuil absolu — 0% vs T-bills

Antonacci utilise 0% comme proxy des T-bills. Tester un seuil explicite basé sur le taux sans risque.

In [ ]:
def backtest_with_threshold(monthly_prices, lookback=12, abs_threshold=0.0):
    """Dual Momentum with configurable absolute momentum threshold."""
    risky = ['SPY', 'EFA']
    safe = 'BND'
    TC = 0.0005
    
    portfolio_value = [1.0]
    current_holding = None
    
    monthly_ret = monthly_prices.pct_change()
    
    for i in range(lookback + 1, len(monthly_prices)):
        price_now = monthly_prices.iloc[i - 1]
        price_past = monthly_prices.iloc[i - lookback - 1]
        
        returns_12m = {t: (price_now[t] / price_past[t]) - 1 for t in risky}
        best_risky = max(returns_12m, key=returns_12m.get)
        best_return = returns_12m[best_risky]
        
        target = best_risky if best_return > abs_threshold else safe
        
        month_ret = monthly_ret[target].iloc[i]
        tc = TC if target != current_holding else 0
        current_holding = target
        
        pv = portfolio_value[-1] * (1 + month_ret - tc)
        portfolio_value.append(pv)
    
    dates = monthly_prices.index[lookback + 1:]
    return pd.Series(portfolio_value[1:], index=dates)


thresholds = [-0.05, -0.02, 0.0, 0.02, 0.05, 0.10]
threshold_results = {}

for t in thresholds:
    pv = backtest_with_threshold(monthly_prices, lookback=12, abs_threshold=t)
    pv_slice = pv['2015':]
    if len(pv_slice) > 12:
        threshold_results[f'{t:.0%}'] = compute_metrics(pv_slice)

threshold_df = pd.DataFrame(threshold_results).T
threshold_df.index.name = 'Seuil absolu'

print("=== Sensibilité au seuil momentum absolu ===")
display_t = threshold_df.copy()
for col in ['CAGR', 'Volatilité', 'Max DD']:
    display_t[col] = display_t[col].map('{:.2%}'.format)
for col in ['Sharpe', 'Sortino', 'Calmar']:
    display_t[col] = display_t[col].map('{:.3f}'.format)
print(display_t)

## 8. Conclusions et Recommandations

### Résumé des findings

In [ ]:
# Recalcul final pour le résumé
pv_final, signals_final, _ = backtest_dual_momentum(monthly_prices, lookback_months=12)
pv_2015_final = pv_final['2015':]
spy_bh_aligned = (1 + monthly_prices['SPY'].pct_change()['2015':]).cumprod()
spy_bh_aligned = spy_bh_aligned.reindex(pv_2015_final.index, method='ffill')

dm_m = compute_metrics(pv_2015_final)
spy_m = compute_metrics(spy_bh_aligned)

print("=" * 60)
print("RESUME FINAL - Dual Momentum vs SPY (2015-2026)")
print("=" * 60)
print(f"  Dual Momentum: CAGR={dm_m['CAGR']:.2%}, Sharpe={dm_m['Sharpe']:.3f}, MaxDD={dm_m['Max DD']:.2%}")
print(f"  SPY Buy&Hold:  CAGR={spy_m['CAGR']:.2%}, Sharpe={spy_m['Sharpe']:.3f}, MaxDD={spy_m['Max DD']:.2%}")
print()

# Position distribution
pos_dist = signals_final['2015':]['target'].value_counts(normalize=True)
print("Distribution des positions:")
for asset, pct in pos_dist.items():
    print(f"  {asset}: {pct:.1%}")

print()
print("CONFIGURATION RECOMMANDEE pour main.py:")
print("-" * 40)
print("  Lookback: 12 mois (252 jours trading)")
print("  Seuil absolu: 0% (proxy T-bills, Antonacci standard)")
print("  Univers: SPY + EFA + BND")
print("  Rebalance: mensuel (premier jour du mois)")
print("  Position: 100% dans l'actif cible")
print("  Transaction costs: ~5 bps")

In [ ]:
# Configuration recommandée
recommended_config = {
    "lookback_months": 12,     # Robuste: 9-18 mois donne résultats similaires
    "abs_threshold": 0.0,      # 0% proxy T-bills (Antonacci standard)
    "risky_assets": ["SPY", "EFA"],  # US + Intl equities
    "safe_asset": "BND",       # Bonds comme refuge
    "rebalance": "monthly",    # Mensuel suffit, moins de frais
    "position_size": 1.0,      # 100% concentration (comme Antonacci)
    "start_date": "2015-01-01",
    "end_date": "2026-01-01",
}

print("Configuration QC recommandée:")
for k, v in recommended_config.items():
    print(f"  {k}: {v}")